# 18 - Getting started: run the stable SJD video demo

This operator notebook demonstrates the stable Spark Job Definition workflow with three sample videos:

1. register immutable work;
2. atomically claim one compatible batch;
3. process the batch with the reviewed ONNX CPU profile;
4. publish stable `people_counter_sjd_gold_*` analytics tables;
5. frame the Direct Lake semantic model; and
6. open the Power BI report.

> **Safety:** every side-effect gate is `False` by default. Run the parameter, input, and helper cells first, then enable and run one stage at a time.

Attach the `people_counter_dev` Lakehouse and the published people-counter Environment before running this notebook. Presenter instructions are in [`18_sjd_getting_started_demo_tutorial.md`](./18_sjd_getting_started_demo_tutorial.md).

In [ ]:
WORKSPACE_ID = "c31ee864-230d-4005-8fd5-7c7130ebf774"
LAKEHOUSE_ID = "883cff91-eaa8-40be-870f-6e9716303cb2"
CONTROL_SJD_ID = "6df10e00-517c-409a-8dd6-40ae9bc62003"
PROCESS_SJD_ID = "c6397f52-176d-40eb-ab8e-c37844fd90d6"
GOLD_SJD_ID = "621391c9-ba0c-4965-8f1e-52fbb46b59eb"
SEMANTIC_MODEL_ID = "9e6b8041-44a2-4f46-9ad8-85b7ca87b71e"
REPORT_ID = "c471e831-3a3e-4828-ad5d-720d4111efe1"
REPORT_PAGE_ID = "361e613179612be95d9c"

# Leave blank to create a UTC timestamped run label. Set this once when you
# need to resume the same demo after restarting the notebook session.
RUN_LABEL = ""

# Enable and run exactly one stage at a time.
REGISTER_INPUTS = False
CLAIM_BATCH = False
PROCESS_BATCH = False
BUILD_GOLD = False
REFRESH_MODEL = False
VERIFY_RESULTS = False

POLL_SECONDS = 15
JOB_TIMEOUT_SECONDS = 3600

## 1. Reviewed demo inputs and release identity

The source and model hashes are part of the registration payload. The process job independently hashes the localized files and fails closed on drift. The three durations total approximately **106.86 seconds**, or **0.03 video-hours**.

In [ ]:
VIDEOS = [
    {
        "video_uri": "abfss://c31ee864-230d-4005-8fd5-7c7130ebf774@onelake.dfs.fabric.microsoft.com/883cff91-eaa8-40be-870f-6e9716303cb2/Files/videos/incoming/2023/09/26/c82ea5ff393efd0de50e7d2fa9f0620829b39caee5dd3e5f92b684ffe74e0f9c/4a780417943542a7d0e72d4f91f30751b8fad304f911936ce7c07ee5a49584c5/people_crossing.mp4",
        "sample_name": "people_crossing",
        "duration_seconds": 68.466667,
        "source_sha256": "4a780417943542a7d0e72d4f91f30751b8fad304f911936ce7c07ee5a49584c5",
    },
    {
        "video_uri": "abfss://c31ee864-230d-4005-8fd5-7c7130ebf774@onelake.dfs.fabric.microsoft.com/883cff91-eaa8-40be-870f-6e9716303cb2/Files/videos/incoming/2023/09/29/5b300380f4ea792bf57f1ee1ff6a015258b9da63f5ccdd34bdfbcfb8d3564468/d5bd760178a4ae0d429a52c2c46ae6ec567fea2d55608e982293bfc001a8f750/subway.mp4",
        "sample_name": "subway",
        "duration_seconds": 31.396667,
        "source_sha256": "d5bd760178a4ae0d429a52c2c46ae6ec567fea2d55608e982293bfc001a8f750",
    },
    {
        "video_uri": "abfss://c31ee864-230d-4005-8fd5-7c7130ebf774@onelake.dfs.fabric.microsoft.com/883cff91-eaa8-40be-870f-6e9716303cb2/Files/videos/incoming/2023/10/01/cbc645f15a90197b2fb3c6d43eed72085c0d32f6ed9acd0cb00826ab04e445b9/0e70fed4fcdc59334f16dc1eac1a5f433403af4fb53a348a26d24b07404b58d4/three_people_walking.mp4",
        "sample_name": "three_people_walking",
        "duration_seconds": 7.0,
        "source_sha256": "0e70fed4fcdc59334f16dc1eac1a5f433403af4fb53a348a26d24b07404b58d4",
    },
]

PROFILE_ID = "onnx-r18-b1-1fps-1t"
PROFILE_SHA256 = "2b816ded6660744bb8dc96e26cd284c0bd78105b905b322b01b6b8acaec96c37"
RELEASE_MANIFEST_PATH = "Files/people-counter/sjd/v1/releases/0.9.49/detached-manifest.json"
RELEASE_MANIFEST_SHA256 = "d04e610a4e7dadf33dd6b6580b0e2c6c2275ca85c3af1e355c5342c7771ca299"
RELEASE_RECEIPT_PATH = "Files/people-counter/sjd/v1/releases/0.9.49/postpublish-receipt.json"
RELEASE_RECEIPT_SHA256 = "0563c0f6f308d2926801082291c3f72196676cbbcd814dd626f8868489c7f26f"
MODEL_ARTIFACT_SHA256 = {
    "rtdetr_osnet/rtdetr_v2_r18vd/config.json": "ed051ec77cb41c5d9d5e3af21a979b1e890599dfd68434d7636ff82ded4c1527",
    "rtdetr_osnet/rtdetr_v2_r18vd/preprocessor_config.json": "cd38cd59999e7a95d68e487fbe5132df3d4e5c32a0836add57e6126ba0c4eaf1",
    "rtdetr_osnet/rtdetr_v2_r18vd/model.onnx": "425192fda18c29867123479a7a047cec0797d24a6a68ebbcad1faaa702c00ef3",
    "rtdetr_osnet/libre_reid_osnet/osnet_ain_x0_25.onnx": "da73234e40324fe032847368dccc2aab7297407ac98f2463ff9395b09434b64a",
}

assert len(VIDEOS) == 3
assert round(sum(video["duration_seconds"] for video in VIDEOS) / 3600, 2) == 0.03
print(f"Reviewed {len(VIDEOS)} videos ({sum(v['duration_seconds'] for v in VIDEOS):.2f} seconds)")

## 2. Fabric REST and polling helpers

The notebook obtains short-lived tokens from `notebookutils`; it never stores credentials. It invokes only the fixed stable SJD item IDs above and treats every non-completed terminal state as a failure.

In [ ]:
from datetime import datetime, timezone
import base64
import json
import re
import time

import notebookutils
import requests
from pyspark.sql import functions as F


FABRIC_API = "https://api.fabric.microsoft.com/v1"
POWER_BI_API = "https://api.powerbi.com/v1.0/myorg"
TERMINAL_JOB_STATES = {"Completed", "Failed", "Cancelled", "Deduped"}
run_label = RUN_LABEL.strip() or datetime.now(timezone.utc).strftime("demo-%Y%m%d-%H%M%S")
if re.fullmatch(r"[a-z0-9-]+", run_label) is None:
    raise ValueError("RUN_LABEL must contain only lowercase letters, digits, and hyphens")
WORK_IDS = [f"{run_label}-{video['sample_name'].replace('_', '-')}" for video in VIDEOS]
OWNER = f"{run_label}-owner"


def fabric_headers():
    token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}


def power_bi_headers():
    token = notebookutils.credentials.getToken("pbi")
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}


def invoke_sjd(item_id, arguments):
    url = f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/sparkJobDefinitions/{item_id}/jobs/sparkjob/instances"
    response = requests.post(
        url,
        headers=fabric_headers(),
        json={"executionData": {"commandLineArguments": arguments}},
        timeout=60,
    )
    if response.status_code != 202:
        raise RuntimeError(f"SJD invocation failed ({response.status_code}): {response.text}")
    body = response.json() if response.content else {}
    job_id = body.get("id") or body.get("jobInstanceId")
    if not job_id:
        job_id = response.headers.get("Location", "").rstrip("/").split("/")[-1]
    if not job_id:
        raise RuntimeError("Fabric returned no SJD job instance ID")
    print(f"Triggered {item_id}: {job_id}")
    return job_id


def wait_for_sjd(item_id, job_id):
    url = f"{FABRIC_API}/workspaces/{WORKSPACE_ID}/items/{item_id}/jobs/instances/{job_id}"
    deadline = time.monotonic() + JOB_TIMEOUT_SECONDS
    while time.monotonic() < deadline:
        response = requests.get(url, headers=fabric_headers(), timeout=60)
        if response.status_code != 200:
            raise RuntimeError(f"SJD status failed ({response.status_code}): {response.text}")
        state = response.json()
        status = state.get("status")
        print(f"{job_id}: {status}")
        if status in TERMINAL_JOB_STATES:
            if status != "Completed":
                raise RuntimeError(json.dumps(state, indent=2, sort_keys=True))
            return state
        time.sleep(POLL_SECONDS)
    raise TimeoutError(f"SJD job {job_id} did not finish within {JOB_TIMEOUT_SECONDS} seconds")


def payload_base64(payload):
    encoded = json.dumps(payload, allow_nan=False, separators=(",", ":"), sort_keys=True).encode()
    return base64.urlsafe_b64encode(encoded).decode("ascii")


def registration_payload(video, captured_at_utc):
    return {
        "batch_size": 1,
        "camera_id": "demo-camera",
        "camera_timezone": "UTC",
        "captured_at_utc": captured_at_utc,
        "detector_model": "r18",
        "device_variant": "cpu",
        "location_id": "demo-location",
        "model_artifact_sha256": MODEL_ARTIFACT_SHA256,
        "model_format": "onnx",
        "pipeline": "rtdetr-osnet",
        "profile_id": PROFILE_ID,
        "runtime_key": PROFILE_SHA256,
        "sample_fps": 1.0,
        "sample_name": video["sample_name"],
        "source_sha256": video["source_sha256"],
        "source_video": video["video_uri"],
    }


print(f"Run label: {run_label}")
print("Work IDs:")
for work_id in WORK_IDS:
    print(f"  {work_id}")

## 3. Register the videos

Preserve `RUN_LABEL` if the notebook session will be restarted.

In [ ]:
captured_at_utc = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")
registration_arguments = []
for work_id, video in zip(WORK_IDS, VIDEOS):
    payload = registration_payload(video, captured_at_utc)
    arguments = " ".join(
        [
            "register",
            f"--work-id {work_id}",
            f"--payload-base64 {payload_base64(payload)}",
            f"--runtime-key {PROFILE_SHA256}",
            f"--duration-seconds {video['duration_seconds']}",
            f"--config-sha256 {PROFILE_SHA256}",
            f"--release-digest {RELEASE_MANIFEST_SHA256}",
            "--max-attempts 3",
        ]
    )
    registration_arguments.append(arguments)

if REGISTER_INPUTS:
    for work_id, arguments in zip(WORK_IDS, registration_arguments):
        print(f"Registering {work_id}")
        wait_for_sjd(CONTROL_SJD_ID, invoke_sjd(CONTROL_SJD_ID, arguments))
else:
    print("Preview only. Set REGISTER_INPUTS=True to register all three inputs.")
    for work_id in WORK_IDS:
        print(f"  QUEUED candidate: {work_id}")

## 4. Atomically claim one batch

In [ ]:
claim_arguments = " ".join(
    [
        "claim",
        f"--owner {OWNER}",
        *(f"--work-id {work_id}" for work_id in WORK_IDS),
        "--max-items 3",
        "--minimum-items 3",
        "--lease-seconds 3600",
        "--minimum-speed-x 0.1",
        "--safety-factor 2",
        "--margin-seconds 60",
    ]
)

BATCH_ID = ""
if CLAIM_BATCH:
    wait_for_sjd(CONTROL_SJD_ID, invoke_sjd(CONTROL_SJD_ID, claim_arguments))
    rows = (
        spark.table("people_counter_sjd_batches")
        .where(F.col("owner") == OWNER)
        .orderBy(F.col("created_at").desc())
        .select("batch_id", "owner", "status", "item_count", "created_at")
        .limit(1)
        .collect()
    )
    if len(rows) != 1 or rows[0]["item_count"] != 3:
        raise RuntimeError(f"Expected one three-item batch for {OWNER!r}; observed {rows!r}")
    BATCH_ID = rows[0]["batch_id"]
    display(spark.createDataFrame(rows))
    print(f"Batch ID: {BATCH_ID}")
else:
    print("Preview only. Set CLAIM_BATCH=True after registration completes.")
    print(claim_arguments)

## 5. Process the claimed batch

If the notebook session was restarted after claiming, set `BATCH_ID` below to the value printed by the claim stage.

In [ ]:
# Set this only when resuming a previously claimed batch in a new session.
RESUME_BATCH_ID = ""
selected_batch_id = BATCH_ID or RESUME_BATCH_ID.strip()
process_arguments = " ".join(
    [
        f"--batch-id {selected_batch_id or '<BATCH_ID>'}",
        "--mode sdk",
        f"--release-manifest-path {RELEASE_MANIFEST_PATH}",
        f"--release-manifest-sha256 {RELEASE_MANIFEST_SHA256}",
        f"--release-receipt-path {RELEASE_RECEIPT_PATH}",
        f"--release-receipt-sha256 {RELEASE_RECEIPT_SHA256}",
    ]
)

if PROCESS_BATCH:
    if not selected_batch_id:
        raise ValueError("Claim a batch or set RESUME_BATCH_ID before processing")
    wait_for_sjd(PROCESS_SJD_ID, invoke_sjd(PROCESS_SJD_ID, process_arguments))
else:
    print("Preview only. Set PROCESS_BATCH=True after the claim completes.")
    print(process_arguments)

## 6. Publish stable gold analytics

In [ ]:
gold_arguments = "run --lookback-hours 48"
if BUILD_GOLD:
    wait_for_sjd(GOLD_SJD_ID, invoke_sjd(GOLD_SJD_ID, gold_arguments))
else:
    print("Preview only. Set BUILD_GOLD=True after processing completes.")
    print(gold_arguments)

## 7. Frame the Direct Lake semantic model

In [ ]:
def refresh_semantic_model():
    url = f"{POWER_BI_API}/groups/{WORKSPACE_ID}/datasets/{SEMANTIC_MODEL_ID}/refreshes"
    response = requests.post(
        url,
        headers=power_bi_headers(),
        json={"notifyOption": "NoNotification"},
        timeout=60,
    )
    if response.status_code != 202:
        raise RuntimeError(f"Semantic-model refresh failed ({response.status_code}): {response.text}")
    deadline = time.monotonic() + 300
    while time.monotonic() < deadline:
        status_response = requests.get(
            f"{url}?$top=1",
            headers=power_bi_headers(),
            timeout=60,
        )
        if status_response.status_code != 200:
            raise RuntimeError(
                f"Refresh status failed ({status_response.status_code}): {status_response.text}"
            )
        latest = status_response.json()["value"][0]
        status = latest["status"]
        print(f"Semantic model: {status}")
        if status in {"Completed", "Failed", "Disabled"}:
            if status != "Completed":
                raise RuntimeError(json.dumps(latest, indent=2, sort_keys=True))
            return latest
        time.sleep(2)
    raise TimeoutError("Semantic-model refresh did not finish within 300 seconds")


if REFRESH_MODEL:
    refresh_semantic_model()
else:
    print("Preview only. Set REFRESH_MODEL=True after the gold job completes.")

## 8. Verify the result

In [ ]:
if VERIFY_RESULTS:
    result = (
        spark.table("people_counter_sjd_gold_video")
        .where(F.col("work_id").isin(WORK_IDS))
        .select(
            "work_id",
            "video_duration_seconds",
            "processing_seconds",
            "speed_x_realtime",
            "distinct_people",
            "line_in_count",
            "line_out_count",
            "completed_at",
        )
        .orderBy("work_id")
    )
    if result.count() != 3:
        raise RuntimeError("Expected all three demo work IDs in stable gold")
    display(result)
else:
    print("Set VERIFY_RESULTS=True after gold publication and semantic-model refresh.")

REPORT_URL = (
    f"https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}/reports/"
    f"{REPORT_ID}/{REPORT_PAGE_ID}?experience=fabric-developer"
)
print(f"Video processing report: {REPORT_URL}")